# Sentinel-1 and Sentinel-2 parcel zonal statistics

Configure and run `OpenEOJobManagerZonalStats` through the multi-user scheduler. openEO creates sensor-preprocessed physical-band NetCDF cubes per spatial batch. IQR cleaning, null filling, pixel indices, and parcel statistics run locally, producing one GeoParquet row per parcel with dated features and a separate annual summary. See [the detailed workflow](../../../docs/data_preparation/openeo/WORKFLOW.md).

In [ ]:
from pathlib import Path
import os
from IPython.display import display
import geopandas as gpd

from data_preparation.parcel_stats import estimate_utm_epsg_from_parcels
from data_preparation.parcel_stats.job_manager import compute_tile_buffer_metres
from data_preparation.parcel_stats.multiuser import (
    load_openeo_users_from_db,
    merge_and_save_results,
    run_parallel_extractions,
)
from shared.io import read_data, write_data, file_exists
from shared.spatial_statistics import spatial_clustering_using_buffer
import shared.geometry as geom_l

from data_preparation.features import (
    append_nearest_lake_features,
    append_parcel_elevation,
    append_soil_feature,
    create_dem_mosaic,
    find_dem_tiles,
    validate_projected_metric_crs,
)

## Inputs

Processing order:

`Sentinel-2 L2A`
`acquisition cloud mask`
`optional temporal reduction, or retain acquisitions`
`per-band, per-time-slice raster IQR outlier removal when enabled`
`local null filling`
`local pixel indices from cleaned physical bands`
`parcel statistics`
`one-row-per-parcel time-series pivot`
`ML-ready GeoParquet`

Temporal filling is past-only so later observations cannot leak into earlier feature periods.

In [ ]:
# ============================================================================
# STEP 1: Read data and re-project
# ============================================================================
# Read the data and re-project to the desired EPSG

WORK_PATH = Path(os.getenv("WORK_PATH"))
RUN_IDENTIFIER = "neuro_2024-2025"

PARCELS_PATH = WORK_PATH / RUN_IDENTIFIER / "sample_parcels.csv"
gdf_parcels = read_data(str(PARCELS_PATH))
print("Initial parcels data:")
display(gdf_parcels.head())

print(f"Parcels before dropping duplicate geometries: {len(gdf_parcels)}")
gdf_parcels = gdf_parcels.drop_duplicates(subset=["geom_wkt"]).copy()
print(f"Parcels after dropping duplicate geometries: {len(gdf_parcels)}")

gdf_parcels = gpd.GeoDataFrame(gdf_parcels, geometry=gpd.GeoSeries.from_wkt(gdf_parcels["geom_wkt"]), crs="EPSG:2100")
print(f"Number of parcels: {len(gdf_parcels)}")
gdf_parcels = geom_l.return_valid_geometries(gdf_parcels)
gdf_parcels = geom_l.convert_multipolygons_to_polygons(gdf_parcels)
print(f"Number of parcels after ensuring valid geometries and converting multipolygons to polygons: {len(gdf_parcels)}")
gdf_parcels = gdf_parcels.drop(columns=["geom_wkt"])

display(gdf_parcels)

In [ ]:
print(gdf_parcels.crs)
WORKING_EPSG = gdf_parcels.crs.to_epsg()
# WORKING_EPSG = estimate_utm_epsg_from_parcels(gdf_parcels)
# WORKING_EPSG

In [ ]:
# ============================================================================
# Define the class column and parcel ID column names.
# These are used in the extraction process to identify the class of each parcel
# and its unique identifier.
# ============================================================================
class_column = "crop_category"
parcel_id_column = "parcel_id"

In [ ]:
gdf_parcels = gdf_parcels[gdf_parcels[class_column] != "jjjj"]
label_counts = gdf_parcels[class_column].value_counts()
print(f"Number of unique labels: {len(label_counts)}")
print(gdf_parcels[class_column].value_counts())
print(50*"-")
rare_labels = label_counts[label_counts < 10].index
gdf_parcels.loc[gdf_parcels[class_column].isin(rare_labels), class_column] = "9999"
print(f"Number of unique labels after grouping rare labels: {len(gdf_parcels[class_column].value_counts())}")
print(gdf_parcels[class_column].value_counts())

cond = gdf_parcels[class_column] == "9999"
print(50*"-")
print(f"Excluded parcels with label '9999': {cond.sum()} out of {len(gdf_parcels)}")
gdf_parcels = gdf_parcels[~cond].copy()
print(f"Selected parcels after excluding label '9999': {len(gdf_parcels)}")


PARCELS_PATH = WORK_PATH / RUN_IDENTIFIER / "cleaned_grouped_parcels.gpkg"
write_data(gdf_parcels, str(PARCELS_PATH))

In [ ]:
SOIL_PATH = Path("C:/work_dir/Agro_Soil_Clean.gpkg")
SOIL_LAYER = "agrosoil"

gdf_parcels["x"] = gdf_parcels.centroid.x
gdf_parcels["y"] = gdf_parcels.centroid.y

if file_exists(str(SOIL_PATH)):
    gdf_parcels = append_soil_feature(
        parcels_path=PARCELS_PATH,
        soil_path=SOIL_PATH,
        output_path=PARCELS_PATH,
        soil_layer=SOIL_LAYER,
        attribute_column="GAIOIKANOT",
        overwrite=True,
    )
    assert gdf_parcels.crs == WORKING_EPSG

PARCELS_PATH = WORK_PATH / RUN_IDENTIFIER / "cleaned_featured_parcels.gpkg"
write_data(gdf_parcels, str(PARCELS_PATH))

In [ ]:
# Reuse saved cluster labels; generate them only when the input has none.
CLUSTER_FIELD = "cluster"
if CLUSTER_FIELD in gdf_parcels.columns:
    gdf_grouped = gdf_parcels.copy()
else:
    gdf_grouped = spatial_clustering_using_buffer(
        gdf_parcels.to_crs(epsg=WORKING_EPSG),
        parcel_id_column,
        min_cluster_distance_in_m=20_000,
        field_cluster=CLUSTER_FIELD,
        plot_results=False,
    )

# groupby would silently omit parcels with missing labels.
if gdf_grouped[CLUSTER_FIELD].isna().any():
    raise ValueError("Every parcel must have a cluster label before partitioning.")

PARCELS_PATH = WORK_PATH / RUN_IDENTIFIER / "cleaned_grouped_featured_parcels.gpkg"
write_data(gdf_grouped, str(PARCELS_PATH))

## Multi-user extraction planning

Accounts are loaded from PostgreSQL using `load_openeo_users_from_db(app_user_id=APP_USER_ID)`. Set the owner ID
in the next cell or provide `APP_USER_ID` in the kernel environment. Install the `database` and `credentials` extras.
Provide `DB_NAME` and `POSTGRES_*` in the kernel environment. The account cell reads the web app's
`SATELLITES_CREDENTIAL_KEY` from `.env` in the working directory or its parents, without changing database settings.
Install `python-dotenv` in the notebook kernel if needed (`%pip install python-dotenv`); never print the key.
`TBL_APP_USERS` and `TBL_EO_USERS` default to `public.app_users` and `public.eo_users`. `TBL_USERS` is ignored.
Only the active owner's accounts are read; passwords are decrypted in memory. Do not print or save credentials.
Accounts are sorted by username. Explicitly restore the original account order when resuming legacy jobs
without saved owners or using partition scheduling. See [database account setup](../../../docs/data_preparation/openeo/CREDENTIALS.md).

Create one spatial partition per cluster, then plan all tile batches before
submission. A shared queue lets every account process batches from any partition. With `fit_tiles_to_parcels=True`, a cube covers the
group's complete bounding box plus `TILE_BUFFER_METRES` on every side:

- width = east - west + 2 * buffer
- height = north - south + 2 * buffer

Only if either dimension exceeds `TILE_SIZE_METRES` does the planner split the
group spatially, then fit and check each subgroup in the same way. Parcels stay
whole and belong to exactly one job. A single parcel that exceeds the limit with
its buffer raises an error before submission.

A large partition can use all available accounts at once. Each account stays
within OPENEO_JOBS_PER_USER. Existing cubes are reused; previously submitted
jobs resume on their owning account. Keep the original account order when
first resuming a legacy run, which did not record account names.

Partition outputs and the merge step stay the same. Local statistics run after
the shared download queue completes, using LOCAL_BATCH_WORKERS per active
partition (up to one active partition per account).


In [ ]:
# ============================================================================
# STEP 2: Load accounts for the shared batch queue
# ============================================================================
# Accounts share unsubmitted batches from all cluster partitions.

from dotenv import dotenv_values, find_dotenv

# Refresh only the encryption key; keep the notebook's existing database connection settings.
credential_env_file = find_dotenv(usecwd=True)
if credential_env_file:
    credential_key = dotenv_values(credential_env_file).get("SATELLITES_CREDENTIAL_KEY")
    if credential_key is not None:
        os.environ["SATELLITES_CREDENTIAL_KEY"] = credential_key
    del credential_key

OPENEO_JOBS_PER_USER = 2  # This is default by openEO
APP_USER_ID = 3  # Selected app_users owner; use None to read APP_USER_ID from the kernel environment.
if APP_USER_ID is None:
    owner_setting = os.getenv("APP_USER_ID", "").strip()
    if not owner_setting.isdecimal() or int(owner_setting) <= 0:
        raise ValueError("Error: Set APP_USER_ID to your positive app_users.app_user_id before loading accounts.")
    APP_USER_ID = int(owner_setting)
users_list = load_openeo_users_from_db(app_user_id=APP_USER_ID)
print(f"Loaded {len(users_list)} openEO user(s):")
for i, (user, _) in enumerate(users_list, 1):
    print(f"  User {i}: {user}")

In [ ]:
# Keep each cluster together; the existing tile planner handles its size limit.
spatial_parts = [
    part.copy()
    for _, part in gdf_grouped.groupby(CLUSTER_FIELD, sort=True, observed=True)
]
assert sum(len(part) for part in spatial_parts) == len(gdf_grouped)

print(f"Created {len(spatial_parts)} cluster partitions:")
for i, part in enumerate(spatial_parts, 1):
    print(f"  Partition {i}: {part[CLUSTER_FIELD].iloc[0]} - {len(part)} parcels")


In [ ]:
# Maximum requested cube width AND height, including the buffer.
# Each group gets its own fitted extent; only oversized groups are split.
TILE_SIZE_METRES = 20_000
TILE_BUFFER_METRES = compute_tile_buffer_metres(gdf_grouped, WORKING_EPSG)

print(f"Maximum buffered cube width/height: {TILE_SIZE_METRES:,} m")
print(f"Buffer on each side: {TILE_BUFFER_METRES:,} m")


In [ ]:
# ============================================================================
# Confirm the planned global remote concurrency.
# ============================================================================
TOTAL_REMOTE_JOB_CAPACITY = len(users_list) * OPENEO_JOBS_PER_USER
print(
    f"Remote capacity: {len(users_list)} users x {OPENEO_JOBS_PER_USER} jobs "
    f"= {TOTAL_REMOTE_JOB_CAPACITY} simultaneous jobs"
)
